In [1]:
from ingest import load_faq_data
documents = load_faq_data()

In [2]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

113

In [3]:
documents = documents_llm

In [4]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [5]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [6]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [7]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [8]:
import json

user_prompt = json.dumps(doc)

In [9]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [10]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [11]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [12]:
result = response.output_parsed

print(result)

questions=['I just found this course — can I still start it now, or is it too late to join?', 'If I join the course late, will I still be able to get a certificate?', 'What do I need to do to qualify for the certificate if I’m starting after the course already began?', 'Is there a deadline for project submission if I want the certificate?', 'Can I still participate in the course, and what happens if project submissions are already closed?']


In [13]:
print(result.questions)

['I just found this course — can I still start it now, or is it too late to join?', 'If I join the course late, will I still be able to get a certificate?', 'What do I need to do to qualify for the certificate if I’m starting after the course already began?', 'Is there a deadline for project submission if I want the certificate?', 'Can I still participate in the course, and what happens if project submissions are already closed?']


In [15]:
from evaluation_utils import llm_structured

result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['Can I still start the course if I just found out about it, or is it too late to join?', 'If I join the course now, will I still be able to get a certificate?', 'What do I need to do to qualify for the certificate if I enter the course late?', 'Is there a deadline for project submission if I want the course certificate?', "If I'm joining after the course has already started, can I still submit the project and receive certification?"]


In [16]:
usage.input_tokens, usage.output_tokens

(207, 107)

In [17]:
from evaluation_utils import calc_price

cost = calc_price(usage)

cost

{'input_cost': 0.00015525, 'output_cost': 0.0004815, 'total_cost': 0.00063675}

In [18]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Can I still start the course if I just found out about it, or is it too late to join?',
  'document': '74eb249bbf'},
 {'question': 'If I join the course now, will I still be able to get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to qualify for the certificate if I enter the course late?',
  'document': '74eb249bbf'},
 {'question': 'Is there a deadline for project submission if I want the course certificate?',
  'document': '74eb249bbf'},
 {'question': "If I'm joining after the course has already started, can I still submit the project and receive certification?",
  'document': '74eb249bbf'}]